# 03 — Walk-forward PCA сигналы (отладка одного окна)

Запускает один шаг rolling-fit пайплайна для ручной проверки сигналов и параметров.

**Что делает:**
- Загружает canonical panel и стратегический конфиг
- Вычисляет простые доходности через `compute_simple_returns`
- Инициализирует `RollingFitPipeline` с параметрами из конфига: число PCA-компонент, окно оценки остатков, сдвиг сигнала, `dt_per_year`, пороги входа/выхода
- Берёт первое временное окно (`iter_windows`) и прогоняет `fit_window` для одного актива — выводит промежуточный результат

**Когда использовать:** при отладке параметров стратегии, проверке корректности сигналов или изучении поведения модели на конкретном окне. Не запускает полный бэктест.

In [ ]:
from sarb.config import load_market_config, load_strategy_config
from sarb.data.canonical import load_canonical_panel
from sarb.features.returns import compute_simple_returns
from sarb.walkforward.rolling_fit import RollingFitPipeline

market = load_market_config("usa")
strategy = load_strategy_config()
panel = load_canonical_panel(
    market.canonical_path,
    market=market.market,
    frequency=market.frequency,
)
returns = compute_simple_returns(panel.prices.dropna(how="all"))
pipeline = RollingFitPipeline(
    pca_components=strategy.settings["pca"]["n_components"],
    estimation_window=strategy.settings["residuals"]["estimation_window"],
    signal_shift=strategy.settings["signals"]["shift_bars"],
    dt_per_year=strategy.settings["ou"]["dt_per_year"],
    thresholds=strategy.settings["signals"],
)
window = next(pipeline.iter_windows(returns))
pipeline.fit_window(window, asset=returns.columns[0], full_returns=returns)